In [7]:
import pandas as pd
import numpy as np 
import os

print("Current working directory:", os.getcwd())

Current working directory: d:\projects\DAU_projects\ml-assisted-re-distribution\v2_real_H_benchmark


In [10]:
files = {
    "Ba133": "../data/LaBr3 spectrum data/Ba133_calibrated_energy_counts.txt",
    "Co60":  "../data/LaBr3 spectrum data/Co60_calibrated_energy_counts.txt",
    "Cs137": "../data/LaBr3 spectrum data/Cs137_calibrated_energy_counts.txt",
    "Eu152": "../data/LaBr3 spectrum data/Eu152_calibrated_energy_counts.txt",
    "Na22":  "../data/LaBr3 spectrum data/Na22_calibrated_energy_counts.txt",
}

results = [] 

for isotope, path in files.items():
    if not os.path.exists(path):
        print(f"MISSING FILE for {isotope}: {path}")
        continue

    df = pd.read_csv(path, sep = "\t")
    e = df["Energy_keV"].to_numpy()
    c = df["Counts"].to_numpy()

    steps = np.diff(e)
    is_uniform = np.allclose(steps, steps[0], atol=1e-6)
    step_val = steps[0] if is_uniform else "NON-UNIFORM"

    # Count statistics
    total_counts = c.sum()
    max_counts = c.max()
    zero_frac = round((c == 0).sum() / len(c), 3)
    low_count_bins = int(((c >0) & (c < 10)).sum())

    results.append({
        "isotope": isotope,
        "min_keV": e.min(),
        "max_keV": e.max(),
        "step_keV": step_val,
        "n_bins": len(e),
        "total_counts": total_counts,
        "max_counts": max_counts,
        "zero_frac": zero_frac,
        "low_count_bins(1-9)": low_count_bins,
    })


summary = pd.DataFrame(results)

print(summary)

  isotope    min_keV      max_keV  step_keV  n_bins  total_counts  max_counts  \
0   Ba133  11.446639  3547.378687  1.729062    2046        538265       36178   
1    Co60  -1.932124  3645.944227  1.783802    2046        123758        1723   
2   Cs137  11.446639  3547.378687  1.729062    2046        834027       16685   
3   Eu152  -1.932124  3645.944227  1.783802    2046        551944       35507   
4    Na22  11.446639  3547.378687  1.729062    2046         36237        1615   

   zero_frac  low_count_bins(1-9)  
0      0.349                  837  
1      0.310                  591  
2      0.338                  786  
3      0.326                  559  
4      0.337                  690  


In [ ]:
# !pip install spectres

In [14]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.response_matrix import load_response_matrix

matrix = load_response_matrix("../data/response_matrix.csv")

H_d = matrix.h              # shape (6100, 599)
e_meas = matrix.e_meas_keV  # (6100,) — 0.5 to 6099.5 keV
e_true = matrix.e_true_keV  # (599,) — 20 to 6000 keV

print(H_d.shape, e_meas.min(), e_meas.max())

(6100, 599) 0.5 6099.5


In [ ]:
from src.response_matrix import validate_response_matrix
from dataclasses import asdict

baseline = validate_response_matrix(matrix)

for k, v in asdict(baseline).items():
    print(f"{k}: {v}")

shape: (6100, 599)
e_meas_range_keV: (0.5, 6099.5)
e_true_range_keV: (20.0, 6000.0)
e_meas_step_first_keV: 1.0
e_true_step_first_keV: 10.0
finite: True
min_value: 0.0
max_value: 0.52212
negative_count: 0
column_sum_min: 0.9989999999999998
column_sum_max: 1.0000000000000004
column_sum_mean: 0.9999959432387312
column_sum_std: 4.9370698732657376e-05
dead_columns: 0
dead_rows: 9


In [ ]:
from spectres import spectres

H_d = 
e_meas = 

# Group A: Ba133 / Cs137 / Na22 share this grid
group_A_grid = e_Ba133
group_A_step = 1.729062

# Group B: Co60 / Eu152 share this grid
group_B_grid = e_Co60 
group_B_step = 1.783802


def rebin_H(H, e_meas, new_grid, new_step):
    """
    Downsamples H's E_meas axis (rows) onto new_grid, leaving E_true (columns)
    untouched. Returns H reshaped to (len(new_grid), H.shape[1]).
    """
    H_T = H.T   # shape (n_true, n_meas) — SpectRes needs the resampled axis last

    # fill=0.0: physically correct for any new bin with no detector response

    density = spectres(new_grid, e_meas, H_T, fill=0.0, verbose=False)

    # density -> mass: SpectRes returns per-keV density: without this multiply,
    # every value is wrong by a factor of new_step.
    mass = density * new_step

    return mass.T   # back to (n_new, n_true)

H_groupA = rebin_H(H_d, e_meas, group_A_grid, group_A_step)
H_groupB = rebin_H(H_d, e_meas, group_B_grid, group_B_step)

colsum_A = H_groupA.sum(axis=0)
colsum_B = H_groupB.sum(axis=0)

print("Group A column sums — min/max:", colsum_A.min(), colsum_A.max())
print("Group B column sums — min/max:", colsum_B.min(), colsum_B.max())